# 2. Reading & Understanding the Data: Multi-City Satellite Time Series

This notebook loads the parquet datasets for **Madrid** (Stage 1 training) and **Amsterdam** (Stage 2 transfer).
We inspect spectral distributions, temporal observation depths, QA masks, and city-level differences.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Search paths for Parquet files (local and remote compatible) ────────────────
search_paths = [
    '.',
    './data',
    '../data',
    'Building_Age_Classification/data',
    './Building_Age_Classification/data'
]

madrid_path = None
amsterdam_path = None

for p in search_paths:
    m_candidate = os.path.join(p, 'madrid_train.parquet')
    a_candidate = os.path.join(p, 'amsterdam_data.parquet')
    if os.path.exists(m_candidate) and madrid_path is None:
        madrid_path = m_candidate
    if os.path.exists(a_candidate) and amsterdam_path is None:
        amsterdam_path = a_candidate

print(f"Madrid data path: {madrid_path}")
print(f"Amsterdam data path: {amsterdam_path}")

df_madrid = pd.read_parquet(madrid_path)
df_amsterdam = pd.read_parquet(amsterdam_path)

print(f"Madrid dataframe shape: {df_madrid.shape}")
print(f"Amsterdam dataframe shape: {df_amsterdam.shape}")

### 2.2 Dataset Structure & Columns
Each row represents one **geographic pixel** in one **year**.
Columns include:
- Metadata: `px_key`, `py_key`, `year`, `city`, `coverage`, `weighted_mean_year`
- 3 observations per year: `Blue_1`, `Green_1`, `Red_1`, `NIR_1`, `SWIR1_1`, `SWIR2_1`, `qa_valid_1` (and slots 2 and 3).

In [ ]:
for name, df in [('Madrid', df_madrid), ('Amsterdam', df_amsterdam)]:
    n_pixels = df.groupby(['px_key', 'py_key']).ngroups
    years = sorted(df['year'].unique())
    print(f"=== {name} Dataset Overview ===")
    print(f"  Total rows (pixel-years): {len(df):,}")
    print(f"  Unique geographic pixels: {n_pixels:,}")
    print(f"  Years covered: {min(years)} - {max(years)} ({len(years)} total years)")
    print(f"  Building coverage: min={df['coverage'].min():.2f}, max={df['coverage'].max():.2f}, mean={df['coverage'].mean():.2f}")
    print(f"  Weighted mean year: min={df['weighted_mean_year'].min():.1f}, median={df['weighted_mean_year'].median():.1f}, max={df['weighted_mean_year'].max():.1f}\n")

### 2.3 Age Class Distribution
Classes are assigned from `weighted_mean_year` using the historical cutoffs:
- **Madrid:** Class 1 (< 1960), Class 2 (1960–1984), Class 3 (1984–2004), Class 4 (2004–2024)
- **Amsterdam:** Class 1 (< 1945), Class 2 (1945–1984), Class 3 (1984–2004), Class 4 (2004–2024)

In [ ]:
def preview_classes(df, city, split_event):
    pixels = df.drop_duplicates(['px_key', 'py_key']).copy()
    bins = [-np.inf, split_event, 1984, 2004, np.inf]
    pixels['age_class'] = pd.cut(pixels['weighted_mean_year'], bins=bins, labels=[1, 2, 3, 4])
    counts = pixels['age_class'].value_counts().sort_index()
    print(f"{city} Age Class Distribution (Split Event = {split_event}):")
    for c, cnt in counts.items():
        pct = (cnt / len(pixels)) * 100
        print(f"  Class {c}: {cnt:,} pixels ({pct:.1f}%)")
    return counts

mad_counts = preview_classes(df_madrid, 'Madrid', 1960)
ams_counts = preview_classes(df_amsterdam, 'Amsterdam', 1945)

### 2.4 Atmospheric Outlier Threshold
In Notebook 2 of the event organizers, Landsat 7 scene `LE07_L2SP_199023_20030529` produces an extreme Blue band artifact (> 60,000) because cirrus clouds are undetected by standard QA flags.
We verify the `BLUE_MAX = 15,000` filtering criterion.

In [ ]:
BLUE_MAX = 15000

for name, df in [('Madrid', df_madrid), ('Amsterdam', df_amsterdam)]:
    blue_cols = [c for c in df.columns if c.startswith('Blue_')]
    outliers = (df[blue_cols] > BLUE_MAX).any(axis=1).sum()
    print(f"{name}: {outliers:,} rows ({outliers/len(df)*100:.2f}%) contain Blue reflectance > {BLUE_MAX:,}")
print("All outliers will be safely cleaned in UnitModel_3_Preprocessing.")